## 0. Uploading clear data

In [1]:
import pandas as pd
import numpy as np
import json
import urllib.request

from src.clean_demo import clean_demo
from src.clean_drug import clean_drug
from src.clean_reac import clean_reac
from src.clean_outc import clean_outc
from src.clean_indi import clean_indi

from src.utils import rename_columns, normalize_str_values, standardize_unknown, fill_str_nulls, convert_str_to_cat, drop_duplicates

from src.map_atc import map_drugs_to_atc

In [3]:
DATA_PATH = '/Users/welcome/Desktop/csv/faers_ascii_2024q2/ASCII/'

DEMO_FILE = DATA_PATH + 'DEMO24Q2.txt'
DRUG_FILE = DATA_PATH + 'DRUG24Q2.txt'
REAC_FILE = DATA_PATH + 'REAC24Q2.txt'
OUTC_FILE = DATA_PATH + 'OUTC24Q2.txt'
INDI_FILE = DATA_PATH + 'INDI24Q2.txt'

In [4]:
df_demo = pd.read_csv(DEMO_FILE, sep='$', low_memory=False)
df_demo = clean_demo(df_demo)

df_drug = pd.read_csv(DRUG_FILE, sep='$', low_memory=False)
df_drug = clean_drug(df_drug)

df_out = pd.read_csv(OUTC_FILE, sep='$', low_memory=False)
df_out = clean_outc(df_out)

In [142]:
unique_drugs = df_exploded['prod_ai_list'].dropna().unique().tolist()
results = map_drugs_to_atc(unique_drugs)

20 already cached, 5136 to fetch
  50/5136 done
  100/5136 done
  150/5136 done
  200/5136 done
  250/5136 done
  300/5136 done
  350/5136 done
  400/5136 done
  450/5136 done
  500/5136 done
  550/5136 done
  600/5136 done
  650/5136 done
  700/5136 done
  750/5136 done
  800/5136 done
  850/5136 done
  900/5136 done
  950/5136 done
  1000/5136 done
  1050/5136 done
  1100/5136 done
  1150/5136 done
  1200/5136 done
  1250/5136 done
  1300/5136 done
  1350/5136 done
  1400/5136 done
  1450/5136 done
  1500/5136 done
  1550/5136 done
  1600/5136 done
  1650/5136 done
  1700/5136 done
  1750/5136 done
  1800/5136 done
  1850/5136 done
  1900/5136 done
  1950/5136 done
  2000/5136 done
  2050/5136 done
  2100/5136 done
  2150/5136 done
  2200/5136 done
  2250/5136 done
  2300/5136 done
  2350/5136 done
  2400/5136 done
  2450/5136 done
  2500/5136 done
  2550/5136 done
  2600/5136 done
  2650/5136 done
  2700/5136 done
  2750/5136 done
  2800/5136 done
  2850/5136 done
  2900/5136 done
 

In [31]:
unique_names = df_drug['drugname'].dropna().unique().tolist()
results = map_drugs_to_atc(
    unique_names,
    cache_file='data/atc/rxnav_atc_cache_drugname.json'
)

15850 already cached, 4336 to fetch
  50/4336 done
  100/4336 done
  150/4336 done
  200/4336 done
  250/4336 done
  300/4336 done
  350/4336 done
  400/4336 done
  1250/4336 done
  1300/4336 done
  1350/4336 done
  1400/4336 done
  1450/4336 done
  1500/4336 done
  1550/4336 done
  1600/4336 done
  1650/4336 done
  1700/4336 done
  1750/4336 done
  1800/4336 done
  1850/4336 done
  1900/4336 done
  1950/4336 done
  2000/4336 done
  2050/4336 done
  2100/4336 done
  2150/4336 done
  2200/4336 done
  2250/4336 done
  2300/4336 done
  2350/4336 done
  2400/4336 done
  2450/4336 done
  2500/4336 done
  2550/4336 done
  2600/4336 done
  2650/4336 done
  2700/4336 done
  2750/4336 done
  2800/4336 done
  2850/4336 done
  2900/4336 done
  2950/4336 done
  3000/4336 done
  3050/4336 done
  3100/4336 done
  3150/4336 done
  3200/4336 done
  3250/4336 done
  3300/4336 done
  3350/4336 done
  3400/4336 done
  3450/4336 done
  3500/4336 done
  3550/4336 done
  3600/4336 done
  3650/4336 done
  37

In [34]:
with open('data/atc/rxnav_atc_cache_drugname.json', 'r') as f:
    partial_results = json.load(f)

print(len(partial_results))

df_partial = pd.DataFrame(list(partial_results.items()), columns=['drugname', 'atc_class'])

20186


In [36]:
df_partial.isna().sum()

drugname         0
atc_class    15193
dtype: int64

In [33]:
df_drug['atc_final'].isna().sum()
df_drug.shape[0]

1262406

In [37]:
df_drug = df_drug.drop(columns=['atc_by_name', 'atc_by_ai', 'atc_final'], errors='ignore')

df_drug = df_drug.merge(df_by_name, on='drugname', how='left')
df_drug = df_drug.merge(df_by_ai, on='prod_ai', how='left')
df_drug['atc_final'] = df_drug['atc_by_ai'].fillna(df_drug['atc_by_name'])

In [42]:
df_drug.drop(['atc_by_name_x',
 'atc_by_ai_x',
 'atc_by_name_y',
 'atc_by_ai_y',
 'atc_by_name',
 'atc_by_ai',
 'atc_final'], axis=1, inplace=True)

In [43]:
with open('data/atc/rxnav_atc_cache_drugname.json', 'r') as f:
    cache_drugname = json.load(f)

with open('data/atc/rxnav_atc_cache.json', 'r') as f:
    cache_prod_ai = json.load(f)

df_by_name = pd.DataFrame(list(cache_drugname.items()), columns=['drugname', 'atc_by_name'])
df_by_ai = pd.DataFrame(list(cache_prod_ai.items()), columns=['prod_ai', 'atc_by_ai'])

# both lib from json
df_by_name = pd.DataFrame(list(cache_drugname.items()), columns=['drugname', 'atc_by_name'])
df_by_ai = pd.DataFrame(list(cache_prod_ai.items()), columns=['prod_ai', 'atc_by_ai'])

# join with the main df_drug
df_drug = df_drug.merge(df_by_name, on='drugname', how='left')
df_drug = df_drug.merge(df_by_ai, on='prod_ai', how='left')

# prod_ai priority, then by drug name
df_drug['atc_final'] = df_drug['atc_by_ai'].fillna(df_drug['atc_by_name'])

print(df_drug['atc_final'].isna().sum())

142152


In [45]:
df_drug.isna().sum()

primaryid              0
caseid                 0
drug_seq               0
role_cod               0
drugname               0
prod_ai                0
route                  0
dose_form              0
is_primary             0
drugs_per_case         0
atc_by_name       188983
atc_by_ai         240751
atc_final         142152
dtype: int64

In [46]:
url = "https://unmtid-dbs.net/download/DrugCentral/2021_09_01/structures.smiles.tsv"
urllib.request.urlretrieve(url, "data/atc/drugcentral_inn.tsv")

('data/atc/drugcentral_inn.tsv', <http.client.HTTPMessage at 0x33cf553d0>)

In [47]:
df_dc = pd.read_csv('data/atc/drugcentral_inn.tsv', sep='\t', 
                     names=['smiles', 'inchi', 'inchikey', 'id', 'inn', 'cas_rn'])
df_dc['inn'] = df_dc['inn'].str.upper().str.strip()

# how many drugs in INN
missing_drugs = df_drug[df_drug['atc_final'].isna()]['prod_ai'].unique()
found_in_dc = set(missing_drugs) & set(df_dc['inn'].dropna())
print(len(found_in_dc), '/', len(missing_drugs))

246 / 3325


In [48]:
salt_suffixes = [
    ' HYDROCHLORIDE', ' SODIUM', ' SULFATE', ' ACETATE', ' MALEATE',
    ' TARTRATE', ' CITRATE', ' PHOSPHATE', ' MESYLATE', ' FUMARATE',
    ' CALCIUM', ' POTASSIUM', ' DIHYDROCHLORIDE', ' BESYLATE',
    ' HYDROBROMIDE', ' SUCCINATE', ' HEMIHYDRATE', ' MONOHYDRATE',
    ' DIHYDRATE', ' ANHYDROUS', ' TRIHYDRATE'
]

def strip_salt_suffix(name):
    for suffix in salt_suffixes:
        if name.endswith(suffix):
            return name[:-len(suffix)].strip()
    return name

missing_drugs_clean = [strip_salt_suffix(d) for d in missing_drugs]

# RxNav vs DrugCentral INN
all_known = set(cache_prod_ai.keys()) | set(cache_drugname.keys()) | set(df_dc['inn'].dropna())
still_missing = [d for d, orig in zip(missing_drugs_clean, missing_drugs) if d not in all_known]

print(len(still_missing), '/', len(missing_drugs))

674 / 3325


In [49]:
still_missing_names = [orig for d, orig in zip(missing_drugs_clean, missing_drugs) if d not in all_known]
still_missing_names[:30]

['ACETAMINOPHEN\\CAFFEINE\\CARBASPIRIN CALCIUM\\CHLORPHENIRAMINE MALEATE\\PROPOXYPHENE HYDROCHLORIDE',
 'ACETAMINOPHEN\\CAFFEINE\\PROPOXYPHENE HYDROCHLORIDE',
 'PIPERACILLIN SODIUM\\TAZOBACTAM SODIUM',
 'AMPHETAMINE ASPARTATE\\AMPHETAMINE ASPARTATE MONOHYDRATE\\AMPHETAMINE SULFATE\\DEXTROAMPHETAMINE SACCHARATE\\DEXTROAMPHETAMINE SULFATE',
 'SODIUM FERROUS CITRATE',
 'RISEDRONATE SODIUM',
 'PYRITHIONE ZINC\\ZINC\\ZINC CHLORIDE',
 'ACETAMINOPHEN\\OXYCODONE HYDROCHLORIDE',
 'CODEINE PHOSPHATE\\PROMETHAZINE HYDROCHLORIDE',
 'GIMERACIL\\OTERACIL POTASSIUM\\TEGAFUR',
 'HYALURONATE SODIUM',
 'CHOLECALCIFEROL\\ERGOCALCIFEROL',
 'NALOXONE HYDROCHLORIDE\\OXYCODONE HYDROCHLORIDE',
 'MYCOPHENOLATE SODIUM',
 'GUAIFENESIN\\PSEUDOEPHEDRINE HYDROCHLORIDE',
 'NIACINAMIDE\\PYRIDOXINE HYDROCHLORIDE\\RIBOFLAVIN\\THIAMINE HYDROCHLORIDE',
 'DEXPANTHENOL\\NIACINAMIDE\\PYRIDOXINE HYDROCHLORIDE\\RIBOFLAVIN 5^ PHOSPHATE SODIUM\\THIAMINE HYDROCHLORIDE',
 'LORATADINE\\PSEUDOEPHEDRINE SULFATE',
 'CITRIC ACID MONOH

In [51]:
from src.clean_atc import clean_atc

df_atc = pd.read_csv('data/atc/who_atc_ddd_2026_2026-02-23.csv')
df_atc = clean_atc(df_atc)

atc_names = set(df_atc['atc5_description'].dropna())

still_missing_names_clean = [strip_salt_suffix(d) for d in still_missing_names]
found_in_who = set(still_missing_names_clean) & atc_names
print(len(found_in_who), '/', len(still_missing_names))

7 / 674


In [144]:
with open('data/atc/rxnav_atc_cache.json', 'r') as f:
    data = json.load(f)

df_atc_map = pd.DataFrame(list(data.items()), columns=['prod_ai', 'atc_class'])

,prod_ai,atc_class
0,CYCLOSPORINE,Calcineurin inhibitors
1,METHOTREXATE,Folic acid analogues
2,CYCLOPHOSPHAMIDE,Nitrogen mustard analogues
3,VINCRISTINE SULFATE,Vinca alkaloids and analogues
4,CYTARABINE,Combinations of antineoplastic agents


In [146]:
df_atc_map.isna().sum()

prod_ai         0
atc_class    2129
dtype: int64

In [147]:
missing = df_atc_map[df_atc_map['atc_class'].isna()]['prod_ai']
missing.head(30).tolist()

['IRON',
 'CARBASPIRIN CALCIUM',
 'PEGINTERFERON ALFA 2B',
 'LAFUTIDINE',
 'ASPIRIN DL LYSINE',
 'MYCOPHENOLATE MOFETIL',
 'THYMOCYTE IMMUNE GLOBULIN NOS',
 'HESPERIDIN METHYLCHALCONE',
 'UNK',
 'LAPINE T LYMPHOCYTE IMMUNE GLOBULIN',
 'CORTICOSTEROID NOS',
 'MONTMORILLONITE',
 'VITAMIN D NOS',
 'ANTIHISTAMINES NOS',
 'MAGNESIUM',
 'CALCIUM',
 'MINERALS',
 'CANNABIS SATIVA SUBSP INDICA TOP',
 'VITAMINS',
 'UNSPECIFIED INGREDIENT',
 'REBAMIPIDE',
 'ALENDRONATE SODIUM',
 'SODIUM FERROUS CITRATE',
 'TIOTROPIUM BROMIDE MONOHYDRATE',
 'IBANDRONATE SODIUM',
 'RISEDRONATE SODIUM',
 'DIETARY SUPPLEMENT',
 'LEVOLEUCOVORIN CALCIUM',
 'VITAMIN D',
 'FISH OIL']

In [148]:
df_drug['drugname'].nunique()

20186

In [75]:
df = pd.read_csv('/Users/welcome/PycharmProjects/drug-safety-analytics/data/atc/who_atc_ddd_2026_2026-02-23.csv', sep=',')

In [80]:
def select_columns(df):
    """
    Select relevant ATC columns and drop rows without atc5_description.
    """
    df = df.copy()
    df = df.dropna(subset=['atc5_description'])
    df = df[['atc1_description', 'atc2_description', 'atc3_description',
              'atc4_description', 'atc5_code', 'atc5_description']]
    return df

In [81]:
df = select_columns(df)
df = rename_columns(df)
df = normalize_str_values(df)
df = standardize_unknown(df)
df = fill_str_nulls(df)
df = convert_str_to_cat(df)
df = drop_duplicates(df)

In [82]:
df.head()

,atc1_description,atc2_description,atc3_description,atc4_description,atc5_code,atc5_description
0,ALIMENTARY TRACT AND METABOLISM,STOMATOLOGICAL PREPARATIONS,STOMATOLOGICAL PREPARATIONS,CARIES PROPHYLACTIC AGENTS,A01AA01,SODIUM FLUORIDE
1,ALIMENTARY TRACT AND METABOLISM,STOMATOLOGICAL PREPARATIONS,STOMATOLOGICAL PREPARATIONS,CARIES PROPHYLACTIC AGENTS,A01AA02,SODIUM MONOFLUOROPHOSPHATE
2,ALIMENTARY TRACT AND METABOLISM,STOMATOLOGICAL PREPARATIONS,STOMATOLOGICAL PREPARATIONS,CARIES PROPHYLACTIC AGENTS,A01AA03,OLAFLUR
3,ALIMENTARY TRACT AND METABOLISM,STOMATOLOGICAL PREPARATIONS,STOMATOLOGICAL PREPARATIONS,CARIES PROPHYLACTIC AGENTS,A01AA04,STANNOUS FLUORIDE
4,ALIMENTARY TRACT AND METABOLISM,STOMATOLOGICAL PREPARATIONS,STOMATOLOGICAL PREPARATIONS,CARIES PROPHYLACTIC AGENTS,A01AA30,COMBINATIONS


In [83]:
df['atc5_description'].nunique()

4937

In [115]:
df_mix = df_drug[(df_drug['prod_ai'].str.contains(r'\\', regex=True)) & (df_drug['is_primary'])]

In [120]:
df_mix.shape

(30646, 11)

In [135]:
df_mix.groupby('prod_ai', as_index=False).agg(n_cases = ('caseid', 'count'))\
    .sort_values('n_cases', ascending=False).tail(20)

,prod_ai,n_cases
127,AVOBENZONE\OCTOCRYLENE,1
329,DURVALUMAB\TREMELIMUMAB,1
331,ECHINACEA ANGUSTIFOLIA\HOMEOPATHICS\ZINC ACETA...,1
122,AVOBENZONE\HOMOSALATE\OCTISALATE,1
121,AVOBENZONE\HOMOSALATE\OCTINOXATE\OCTOCRYLENE\O...,1
119,AVOBENZONE\HOMOSALATE\OCTINOXATE\OCTISALATE,1
116,ATROPINE\DIPHENOXYLATE,1
351,EPINEPHRINE BITARTRATE\LIDOCAINE HYDROCHLORIDE...,1
107,ASPIRIN\OMEPRAZOLE,1
387,FLUOROURACIL\LEUCOVORIN,1


In [137]:
df_drug[df_drug['drugname'] == 'NUTRINEAL PD4']

,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,route,dose_form,is_primary,drugs_per_case,prod_ai_list
530796,237618231,23761823,3,SS,NUTRINEAL PD4,AMINO ACIDS\CALCIUM CHLORIDE\MAGNESIUM CHLORID...,INTRAPERITONEAL,SOLUTION FOR PERITONEAL DIALYSIS,False,18,"[AMINO ACIDS, CALCIUM CHLORIDE, MAGNESIUM CHLO..."
542911,237660304,23766030,7,SS,NUTRINEAL PD4,AMINO ACIDS\CALCIUM CHLORIDE\MAGNESIUM CHLORID...,INTRAPERITONEAL,SOLUTION FOR PERITONEAL DIALYSIS,False,22,"[AMINO ACIDS, CALCIUM CHLORIDE, MAGNESIUM CHLO..."
820756,238750091,23875009,3,PS,NUTRINEAL PD4,AMINO ACIDS\CALCIUM CHLORIDE\MAGNESIUM CHLORID...,INTRAPERITONEAL,SOLUTION FOR PERITONEAL DIALYSIS,True,3,"[AMINO ACIDS, CALCIUM CHLORIDE, MAGNESIUM CHLO..."


In [136]:
df_mix.groupby(['drugname', 'prod_ai'], as_index=False).agg(n_cases = ('caseid', 'count'))\
    .sort_values('n_cases', ascending=False).tail(20)

,drugname,prod_ai,n_cases
679,OLMESARTAN MEDOXOMIL AND AMLODIPINE BESYLATE A...,AMLODIPINE BESYLATE\HYDROCHLOROTHIAZIDE\OLMESA...,1
298,CYANOCOBALAMIN\TIRZEPATIDE,CYANOCOBALAMIN\TIRZEPATIDE,1
54,AKYNZEO (FOSNETUPITANT\PALONOSETRON HYDROCHLOR...,FOSNETUPITANT\PALONOSETRON\PALONOSETRON HYDROC...,1
673,OBAGI MEDICAL PHYSICAL DEFENSE BROAD SPECTRUM ...,TITANIUM DIOXIDE\ZINC OXIDE,1
671,NYSTATIN AND TRIAMCINOLONE,NYSTATIN\TRIAMCINOLONE ACETONIDE,1
300,CYPROTERONE ACETATE\ETHINYL ESTRADIOL,CYPROTERONE ACETATE\ETHINYL ESTRADIOL,1
669,NUTRINEAL PD4,AMINO ACIDS\CALCIUM CHLORIDE\MAGNESIUM CHLORID...,1
668,NUTRINEAL,AMINO ACIDS\CALCIUM CHLORIDE\MAGNESIUM CHLORID...,1
111,ARTICADENT,ARTICAINE HYDROCHLORIDE\EPINEPHRINE BITARTRATE,1
664,NORTREL,ETHINYL ESTRADIOL\NORETHINDRONE,1


In [134]:
df_drug[df_drug['drugname'] == 'ADALIMUMAB']['prod_ai'].unique()

<StringArray>
['ADALIMUMAB\ISOPROPYL ALCOHOL']
Length: 1, dtype: str

In [99]:
df_drug['prod_ai_list'] = df_drug['prod_ai'].str.split(r'\\', regex=True)

In [101]:
df_drug['prod_ai_list'].explode('prod_ai_list')

0                      CYCLOSPORINE
1                      METHOTREXATE
2                  CYCLOPHOSPHAMIDE
3               VINCRISTINE SULFATE
4                        CYTARABINE
                     ...           
1419833                     CALCIUM
1419834              ERGOCALCIFEROL
1419835        POLYETHYLENE GLYCOLS
1419836           MUPIROCIN CALCIUM
1419837    ALPHA TOCOPHEROL ACETATE
Name: prod_ai_list, Length: 1419838, dtype: str

In [139]:
import requests

def get_rxcui(drug_name):
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
    response = requests.get(url)
    return response.json()

get_rxcui("aspirin")

{'idGroup': {'rxnormId': ['1191']}}